In [ ]:

import os

print("Current Jupyter working directory:")
print(os.getcwd())

print("\nFiles and folders available here:")
print(os.listdir())

print("\nSearching for your preprocessed Excel files...")

for root, dirs, files in os.walk(os.getcwd()):
    for file in files:
        if file.endswith(".xlsx") and "preprocessed" in file.lower():
            print(os.path.join(root, file))


In [ ]:

import pandas as pd
import os

# Your notebook is already inside the datasets folder
DATASET_PATH = "."

dataset_files = {
    "Earphones": "earphones_preprocessed.xlsx",
    "Laptop": "laptop_preprocessed.xlsx",
    "Mobile": "mobile_preprocessed.xlsx",
    "Mouse": "mouse_preprocessed.xlsx",
    "Speaker": "speaker_preprocessed.xlsx"
}

datasets = {}

for name, filename in dataset_files.items():

    path = os.path.join(DATASET_PATH, filename)

    df = pd.read_excel(path)
    datasets[name] = df

    print("\nDataset:", name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

print("\nAll five datasets loaded successfully!")


In [ ]:

combined_data = []

for name, df in datasets.items():

    text_col = next(
        col for col in [
            "processed_text",
            "clean_text",
            "text",
            "Kannada_Translation",
            "sentence"
        ]
        if col in df.columns
    )

    temp = df[[text_col, "sentiment"]].copy()
    temp.columns = ["text", "sentiment"]
    temp["dataset"] = name

    combined_data.append(temp)

combined_df = pd.concat(combined_data, ignore_index=True)

combined_df.dropna(subset=["text", "sentiment"], inplace=True)
combined_df["text"] = combined_df["text"].astype(str).str.strip()
combined_df = combined_df[combined_df["text"] != ""]
combined_df.reset_index(drop=True, inplace=True)

print("Combined dataset shape:", combined_df.shape)


In [ ]:

from sklearn.model_selection import train_test_split

X = combined_df["text"]
y = combined_df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    token_pattern=r"(?u)\b\w+\b"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

models = {
    "Naive Bayes": MultinomialNB(),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42,
        class_weight="balanced"
    ),

    "Linear SVM": LinearSVC(
        random_state=42,
        class_weight="balanced"
    )
}

trained_models = {}
predictions = {}

for name, model in models.items():

    model.fit(X_train_tfidf, y_train)

    trained_models[name] = model
    predictions[name] = model.predict(X_test_tfidf)

    print(name, "trained successfully!")

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import pandas as pd

results = []

for name, y_pred in predictions.items():

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, average="macro", zero_division=0
        ),
        "Macro F1": f1_score(
            y_test, y_pred, average="macro", zero_division=0
        )
    })

results_df = pd.DataFrame(results)

display(results_df.round(4))


